# GOAL: Build Prophet model

hyperparameters to tune:
- seasonality_prior_scale (technically for reducing overfitting)
- fourier_order (should likely be done separately for daily and yearly)
- additive vs multiplicative???
- Trend-related (changepoints)
    - n_changepoints (default = 25)
    - changepoint_prior_scale (default = 0.05)
    - changepoint_range (0.8) (this means "changepoints can occur in the first 80% of data")


In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from PreRun import PreRun, PostRun
from prophet import Prophet
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")
#systems and readers
#system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None),(1283,'inverter'),(1283,'meter')]
#system_reader_pairs = [(10,None), (50,None), (51,None)]
system_reader_pairs = [(50,None), (51,None)]

out_dir_additive_holidays = Path('./prophet_errors/')
if not out_dir_additive_holidays.is_dir():
    out_dir_additive_holidays.mkdir()
out_dir_multiplicative_holidays = Path('./prophet_errors_mult/')
if not out_dir_multiplicative_holidays.is_dir():
    out_dir_multiplicative_holidays.mkdir()
   

In [ ]:
#hyperparameters
# changepoint_prior_scale = [0.1,1]
# n_changepoints = [0,5]
# seasonality_prior_scale = [10,20]
# holidays_prior_scale = [2,5]
changepoint_prior_scale = [0.1,0.5]
n_changepoints = [10]
seasonality_prior_scale = [20]
holidays_prior_scale = [2]

for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f"System {system_id}, {reader_type}")

    # get the data
    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
            
    prerun.fill_missing_hours()

    prerun.add_weather_features_only()

    prerun.good_end_days_naive(7)
    prerun.tts_of_data_using_end_days()
    all_data = prerun.amended_data.copy()
    system_recorded_max = prerun.data['energy'].max()
    #will want to remove the second year of good end days from end_days

    #data needs to be in a specific format: columns 'time' --> 'ds', 'energy' --> 'y'
    all_data.rename(columns = {'time': 'ds', 'energy': 'y'}, inplace=True)

    first_day = prerun.good_days.loc[0, 'date']
    remove_until = first_day + timedelta(days=730)
    train_dates = prerun.train_dates.loc[prerun.train_dates['date'] >= remove_until].reset_index(drop=True)
    n = int(len(train_dates) / 200)
    some_train_dates = train_dates[::n]
    print(system_id, len(train_dates), n)
    #get all train/ho data sets ONCE so it's not redone for each (p,d,q)
    splits = []

    for pred_date in some_train_dates['date']:
        train_mask = (all_data['ds'] < pred_date - timedelta(days=1))
        ho_mask = ((all_data['ds'] >= pred_date) &
                (all_data['ds'] < pred_date + timedelta(days=1)))

        train_data = all_data.loc[train_mask].reset_index(drop=True)
        ho_data = all_data.loc[ho_mask].reset_index(drop=True)

        splits.append((pred_date, train_data, ho_data))

    results_dict = {}
    #need to loop through all possible hyperparameters
    for hyp in product(changepoint_prior_scale, n_changepoints, seasonality_prior_scale, holidays_prior_scale):
        cps = hyp[0]
        nc = hyp[1]
        sps = hyp[2]
        hps = hyp[3]

        print(f"  hyperparameters {hyp}")

        errors=[]        

        for pred_date in splits:
            train_data = pred_date[1]
            ho_data = pred_date[2]
            X_ho = ho_data.drop(columns = ['y'])

            proph = Prophet(
                weekly_seasonality = False,
                yearly_seasonality = True,
                changepoint_prior_scale = cps,
                n_changepoints = nc,
                seasonality_prior_scale = sps,
                holidays_prior_scale = hps
            )
            proph.fit(train_data)
            y_pred = proph.predict(X_ho)['yhat']

            #insurance
            #make sure value between 0 and highest observed max
            y_pred = np.clip(y_pred, 0, system_recorded_max)
            #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
            darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
            y_pred = y_pred*np.array(darkness_mask)

            error = PostRun.custom_error(ho_data['y'], y_pred)
            errors.append(error)
        
        results_dict[f"{hyp}"] = errors


    all_errors_df = pd.DataFrame(results_dict)
    all_errors_df.to_csv(f'prophet_errors/{system_id}_{reader_type}_prophet_errors.csv', index = False)

System 50, None
  hyperparameters (0.1, 0, 20, 2)


12:41:08 - cmdstanpy - INFO - Chain [1] start processing
12:41:09 - cmdstanpy - INFO - Chain [1] done processing
12:41:10 - cmdstanpy - INFO - Chain [1] start processing
12:41:10 - cmdstanpy - INFO - Chain [1] done processing
12:41:11 - cmdstanpy - INFO - Chain [1] start processing
12:41:11 - cmdstanpy - INFO - Chain [1] done processing
12:41:11 - cmdstanpy - INFO - Chain [1] start processing
12:41:12 - cmdstanpy - INFO - Chain [1] done processing
12:41:12 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
12:41:12 - cmdstanpy - INFO - Chain [1] start processing
12:41:46 - cmdstanpy - INFO - Chain [1] done processing
12:41:47 - cmdstanpy - INFO - Chain [1] start processing
12:41:47 - cmdstanpy - INFO - Chain [1] done processing
12:41:47 - cmdstanpy - INFO - Chain [1] start processing
12:41:47 - cmdstanpy - INFO - Chain [1] done processing
12:41:48 - cmdstanpy - INFO - Chain [1] start processing
12:41:48 - 

System 51, None
  hyperparameters (0.1, 0, 20, 2)


15:31:00 - cmdstanpy - INFO - Chain [1] start processing
15:31:00 - cmdstanpy - INFO - Chain [1] done processing
15:31:01 - cmdstanpy - INFO - Chain [1] start processing
15:31:01 - cmdstanpy - INFO - Chain [1] done processing
15:31:01 - cmdstanpy - INFO - Chain [1] start processing
15:31:02 - cmdstanpy - INFO - Chain [1] done processing
15:31:02 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:31:02 - cmdstanpy - INFO - Chain [1] start processing
15:31:06 - cmdstanpy - INFO - Chain [1] done processing
15:31:06 - cmdstanpy - INFO - Chain [1] start processing
15:31:07 - cmdstanpy - INFO - Chain [1] done processing
15:31:07 - cmdstanpy - INFO - Chain [1] start processing
15:31:08 - cmdstanpy - INFO - Chain [1] done processing
15:31:08 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
15:31:08 - cmdstanpy - INFO - Chain [1] 

In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed

# def process_system_hyperparameters(pair, read_path, systems_cleaned, changepoint_prior_scale, 
#                                    n_changepoints, seasonality_prior_scale, holidays_prior_scale):
#     """Process a single system with all hyperparameter combinations"""
#     system_id = pair[0]
#     reader_type = pair[1]
    
#     # Load and prepare data
#     prerun = PreRun(system_id=system_id, meter_or_inverter=reader_type, path=read_path, systems_cleaned=systems_cleaned)
#     prerun.load_data()
#     prerun.fill_missing_hours()
#     prerun.add_weather_features_only()
#     prerun.good_end_days_naive(7)
#     prerun.tts_of_data_using_end_days()
    
#     all_data = prerun.amended_data.copy()
#     system_recorded_max = prerun.data['energy'].max()
    
#     # Prepare data format
#     all_data.rename(columns={'time': 'ds', 'energy': 'y'}, inplace=True)
    
#     #remove second year of end dates (to make prophet happier)
#     first_day = (prerun.good_days.loc[0, 'date'])
#     remove_until = first_day + timedelta(days=730)
#     prerun.train_dates = prerun.train_dates.loc[prerun.train_dates['date']>=remove_until].reset_index(drop=True)
#     n = int(len(prerun.train_dates)/500)
#     some_train_dates = prerun.train_dates[::n]
#     # Create splits
#     splits = []
#     for pred_date in some_train_dates['date']:
#         train_mask = (all_data['ds'] < pred_date - timedelta(days=1))
#         ho_mask = ((all_data['ds'] >= pred_date) & (all_data['ds'] < pred_date + timedelta(days=1)))
#         train_data = all_data.loc[train_mask].reset_index(drop=True)
#         ho_data = all_data.loc[ho_mask].reset_index(drop=True)
#         splits.append((pred_date, train_data, ho_data))
    
#     # Process hyperparameters sequentially
#     results_dict = {}
#     hyperparams = list(product(changepoint_prior_scale, n_changepoints, seasonality_prior_scale, holidays_prior_scale))
    
#     for hyp in hyperparams:
#         cps, nc, sps, hps = hyp
#         errors = []
        
#         for pred_date, train_data, ho_data in splits:
#             X_ho = ho_data.drop(columns=['y'])
            
#             proph = Prophet(
#                 weekly_seasonality=False,
#                 yearly_seasonality=True,
#                 changepoint_prior_scale=cps,
#                 n_changepoints=nc,
#                 seasonality_prior_scale=sps,
#                 holidays_prior_scale=hps
#             )
#             proph.fit(train_data)
#             y_pred = proph.predict(X_ho)['yhat']
            
#             # Insurance checks
#             y_pred = np.clip(y_pred, 0, system_recorded_max)
#             darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int)
#             y_pred = y_pred * np.array(darkness_mask)
            
#             error = PostRun.custom_error(ho_data['y'], y_pred)
#             errors.append(error)
        
#         results_dict[str(hyp)] = errors
    
#     return system_id, reader_type, results_dict

# # Hyperparameters
# changepoint_prior_scale = [0.1, 1]
# n_changepoints = [0]
# seasonality_prior_scale = [10, 20]
# holidays_prior_scale = [2, 5]

# print("Starting parallel processing of systems...")

# # Process all systems in parallel (using ThreadPoolExecutor for notebook compatibility)
# with ThreadPoolExecutor(max_workers=2) as executor:
#     futures = {
#         executor.submit(
#             process_system_hyperparameters,
#             pair,
#             read_path,
#             systems_cleaned,
#             changepoint_prior_scale,
#             n_changepoints,
#             seasonality_prior_scale,
#             holidays_prior_scale
#         ): pair for pair in system_reader_pairs
#     }
    
#     for future in as_completed(futures):
#         system_id, reader_type, results_dict = future.result()
#         print(f"✓ Completed System {system_id}, {reader_type}")
        
#         all_errors_df = pd.DataFrame(results_dict)
#         all_errors_df.to_csv(f'prophet_errors/{system_id}_{reader_type}_prophet_errors.csv', index=False)

# print("All systems processed!")

Starting parallel processing of systems...


20:19:56 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.
20:20:03 - cmdstanpy - ERROR - Chain [1] error: code '1' Operation not permitted
Optimization terminated abnormally. Falling back to Newton.


In [4]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

from joblib import Parallel, delayed

def _run_hyp_combo(hyp, splits, system_recorded_max):
    """Run one hyperparameter combination across all date splits. Executes in a worker process."""
    import logging
    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
    from prophet import Prophet
    from PreRun import PostRun
    import numpy as np

    cps, nc, sps, hps = hyp
    errors = []

    for _, train_data, ho_data in splits:
        X_ho = ho_data.drop(columns=['y'])

        proph = Prophet(
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=cps,
            n_changepoints=nc,
            seasonality_prior_scale=sps,
            holidays_prior_scale=hps
        )
        proph.fit(train_data)
        y_pred = proph.predict(X_ho)['yhat'].values

        y_pred = np.clip(y_pred, 0, system_recorded_max)
        darkness_mask = (~((X_ho['proportion_daytime'] == 0) & (X_ho['global_tilted_irradiance'] == 0))).astype(int).values
        y_pred = y_pred * darkness_mask

        error = PostRun.custom_error(ho_data['y'], y_pred)
        errors.append(error)

    return str(hyp), errors


# Hyperparameters — 2×1×2×2 = 8 combos
# changepoint_prior_scale = [0.1, 1]
# n_changepoints = [0]
# seasonality_prior_scale = [10, 20]
# holidays_prior_scale = [2, 5]
changepoint_prior_scale = [0.1]
n_changepoints = [0]
seasonality_prior_scale = [10, 20]
holidays_prior_scale = [2]

N_WORKERS = 4  # parallelism within each system

for pair in system_reader_pairs:
    system_id, reader_type = pair
    print(f"System {system_id}, {reader_type}")

    prerun = PreRun(system_id=system_id, meter_or_inverter=reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
    prerun.fill_missing_hours()
    prerun.add_weather_features_only()
    prerun.good_end_days_naive(7)
    prerun.tts_of_data_using_end_days()

    all_data = prerun.amended_data.copy()
    system_recorded_max = prerun.data['energy'].max()
    all_data.rename(columns={'time': 'ds', 'energy': 'y'}, inplace=True)

    first_day = prerun.good_days.loc[0, 'date']
    remove_until = first_day + timedelta(days=730)
    train_dates = prerun.train_dates.loc[prerun.train_dates['date'] >= remove_until].reset_index(drop=True)
    n = int(len(train_dates) / 300)
    some_train_dates = train_dates[::n]

    splits = []
    for pred_date in some_train_dates['date']:
        train_mask = all_data['ds'] < pred_date - timedelta(days=1)
        ho_mask = (all_data['ds'] >= pred_date) & (all_data['ds'] < pred_date + timedelta(days=1))
        splits.append((
            pred_date,
            all_data.loc[train_mask].reset_index(drop=True),
            all_data.loc[ho_mask].reset_index(drop=True)
        ))

    hyperparams = list(product(changepoint_prior_scale, n_changepoints, seasonality_prior_scale, holidays_prior_scale))
    print(f"  {len(hyperparams)} hyperparameter combos across {N_WORKERS} workers...")

    results = Parallel(n_jobs=N_WORKERS, backend='loky')(
        delayed(_run_hyp_combo)(hyp, splits, system_recorded_max)
        for hyp in hyperparams
    )

    results_dict = {key: errors for key, errors in results}
    all_errors_df = pd.DataFrame(results_dict)
    all_errors_df.to_csv(f'prophet_errors/{system_id}_{reader_type}_prophet_errors.csv', index=False)
    print(f"  Saved.")

print("All systems done!")

System 50, None
  2 hyperparameter combos across 4 workers...
  Saved.
System 51, None
  2 hyperparameter combos across 4 workers...
  Saved.
All systems done!


In [3]:
#check errors to see if any hyperparam shouldn't be run later

#compare hyperparameters
#System 50
print('System 50, None, Prophet')
errors = pd.read_csv('prophet_errors/50_None_prophet_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#they all seem to be appx the same???

System 50, None, Prophet
recorded system max: 7.072975
  Hyperparameters: (0.1, 0, 20, 2)
     mean: 1.5204893164879802, median: 0.6005330027233894, min: 0.0075126455550933, max: 8.893085526627317, std: 2.222703664167387


In [ ]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

from joblib import Parallel, delayed


def _run_one_hyp(cps, nc, sps, hps, splits, system_recorded_max):
    import logging
    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
    from prophet import Prophet
    from PreRun import PostRun
    import numpy as np

    errors = []
    for _, train_data, ho_data in splits:
        X_ho = ho_data.drop(columns=['y'])
        proph = Prophet(
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=cps,
            n_changepoints=nc,
            seasonality_prior_scale=sps,
            holidays_prior_scale=hps
        )
        proph.fit(train_data)
        y_pred = proph.predict(X_ho)['yhat']
        y_pred = np.clip(y_pred, 0, system_recorded_max)
        darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int)
        y_pred = y_pred * np.array(darkness_mask)
        errors.append(PostRun.custom_error(ho_data['y'], y_pred))

    return (cps, nc, sps, hps), errors


changepoint_prior_scale = [0.1, 0.5]
n_changepoints = [10]
seasonality_prior_scale = [20]
holidays_prior_scale = [2]

for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f"System {system_id}, {reader_type}")

    prerun = PreRun(system_id=system_id, meter_or_inverter=reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
    prerun.fill_missing_hours()
    prerun.add_weather_features_only()
    prerun.good_end_days_naive(7)
    prerun.tts_of_data_using_end_days()
    all_data = prerun.amended_data.copy()
    # float() so workers receive a plain Python float, not a numpy scalar
    system_recorded_max = float(prerun.data['energy'].max())

    all_data.rename(columns={'time': 'ds', 'energy': 'y'}, inplace=True)

    first_day = prerun.good_days.loc[0, 'date']
    remove_until = first_day + timedelta(days=730)
    train_dates = prerun.train_dates.loc[prerun.train_dates['date'] >= remove_until].reset_index(drop=True)
    n = int(len(train_dates) / 200)
    some_train_dates = train_dates[::n]

    # .copy() ensures each slice is a fully independent DataFrame before pickling
    splits = []
    for pred_date in some_train_dates['date']:
        train_mask = (all_data['ds'] < pred_date - timedelta(days=1))
        ho_mask = ((all_data['ds'] >= pred_date) & (all_data['ds'] < pred_date + timedelta(days=1)))
        splits.append((
            pred_date,
            all_data.loc[train_mask].reset_index(drop=True).copy(),
            all_data.loc[ho_mask].reset_index(drop=True).copy()
        ))

    hyperparams = list(product(changepoint_prior_scale, n_changepoints, seasonality_prior_scale, holidays_prior_scale))

    results = Parallel(n_jobs=4, backend='loky')(
        delayed(_run_one_hyp)(cps, nc, sps, hps, splits, system_recorded_max)
        for cps, nc, sps, hps in hyperparams
    )

    # key comes from what the worker actually used, not what was passed in
    results_dict = {f"{hyp}": errors for hyp, errors in results}
    all_errors_df = pd.DataFrame(results_dict)
    all_errors_df.to_csv(f'prophet_errors/{system_id}_{reader_type}_prophet_errors.csv', index=False)
    print(f"  Saved.")

print("Done!")

In [7]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

from joblib import Parallel, delayed

def _run_one_hyp(hyp, splits, system_recorded_max):
    import logging
    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
    from prophet import Prophet
    from PreRun import PostRun
    import numpy as np

    cps, nc, sps, hps = hyp
    errors = []

    for pred_date, train_data, ho_data in splits:
        X_ho = ho_data.drop(columns=['y'])

        proph = Prophet(
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=cps,
            n_changepoints=nc,
            seasonality_prior_scale=sps,
            holidays_prior_scale=hps
        )
        proph.fit(train_data)
        y_pred = proph.predict(X_ho)['yhat']

        y_pred = np.clip(y_pred, 0, system_recorded_max)
        darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int)
        y_pred = y_pred * np.array(darkness_mask)

        error = PostRun.custom_error(ho_data['y'], y_pred)
        errors.append(error)

    return f"{hyp}", errors


changepoint_prior_scale = [0.1]
n_changepoints = [10]
seasonality_prior_scale = [20]
holidays_prior_scale = [2]

for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f"System {system_id}, {reader_type}")

    prerun = PreRun(system_id=system_id, meter_or_inverter=reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
    prerun.fill_missing_hours()
    prerun.add_weather_features_only()
    prerun.good_end_days_naive(7)
    prerun.tts_of_data_using_end_days()
    all_data = prerun.amended_data.copy()
    system_recorded_max = prerun.data['energy'].max()

    all_data.rename(columns={'time': 'ds', 'energy': 'y'}, inplace=True)

    first_day = prerun.good_days.loc[0, 'date']
    remove_until = first_day + timedelta(days=730)
    train_dates = prerun.train_dates.loc[prerun.train_dates['date'] >= remove_until].reset_index(drop=True)
    n = int(len(train_dates) / 200)
    some_train_dates = train_dates[::n]

    splits = []
    for pred_date in some_train_dates['date']:
        train_mask = (all_data['ds'] < pred_date - timedelta(days=1))
        ho_mask = ((all_data['ds'] >= pred_date) & (all_data['ds'] < pred_date + timedelta(days=1)))
        train_data = all_data.loc[train_mask].reset_index(drop=True)
        ho_data = all_data.loc[ho_mask].reset_index(drop=True)
        splits.append((pred_date, train_data, ho_data))

    hyperparams = list(product(changepoint_prior_scale, n_changepoints, seasonality_prior_scale, holidays_prior_scale))

    results = Parallel(n_jobs=4, backend='loky')(
        delayed(_run_one_hyp)(hyp, splits, system_recorded_max)
        for hyp in hyperparams
    )

    results_dict = {key: errors for key, errors in results}
    all_errors_df = pd.DataFrame(results_dict)
    all_errors_df.to_csv(f'prophet_errors/{system_id}_{reader_type}_prophet_errors.csv', index=False)
    print(f"  Saved.")

print("Done!")

System 50, None
  Saved.
System 51, None
  Saved.
Done!


In [9]:
#System 10
print('System 10, None, Prophet')
errors = pd.read_csv('prophet_errors/10_None_prophet_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()
#System 50
print('System 50, None, Prophet')
errors = pd.read_csv('prophet_errors/50_None_prophet_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()
#System 51
print('System 51, None, Prophet')
errors = pd.read_csv('prophet_errors/51_None_prophet_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    

System 10, None, Prophet
recorded system max: 1.185825
  Hyperparameters: (0.1, 10, 20, 2)
     mean: 0.029863827886997738, median: 0.0152002897712042, min: 0.0012364961688, max: 0.1827959221335068, std: 0.034978019319069034
  Hyperparameters: (0.5, 10, 20, 2)
     mean: 0.030023904860307246, median: 0.01515807729322775, min: 0.0012298729577971, max: 0.1817054694247255, std: 0.03513911851233923

System 50, None, Prophet
recorded system max: 7.072975
  Hyperparameters: (0.1, 10, 20, 2)
     mean: 1.2535584612518667, median: 0.633533997007382, min: 0.0162326569622438, max: 7.4539729713356, std: 1.5672983377670107

System 51, None, Prophet
recorded system max: 7.2368749999999995
  Hyperparameters: (0.1, 10, 20, 2)
     mean: 0.9606655090359852, median: 0.5450524367040546, min: 0.0616024801351069, max: 5.326777360042748, std: 1.0302791982531103


Now consider making the holidays portion multiplicative instead of additive

In [ ]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
system_reader_pairs = [(10,None), (50,None), (51,None)]


from joblib import Parallel, delayed

def _run_one_hyp(hyp, splits, system_recorded_max):
    import logging
    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
    from prophet import Prophet
    from PreRun import PostRun
    import numpy as np

    cps, nc, sps, hps = hyp
    errors = []

    for pred_date, train_data, ho_data in splits:
        X_ho = ho_data.drop(columns=['y'])

        proph = Prophet(
            weekly_seasonality=False,
            yearly_seasonality=True,
            changepoint_prior_scale=cps,
            n_changepoints=nc,
            seasonality_prior_scale=sps,
            holidays_prior_scale=hps,
            holidays_mode = 'multiplicative'
        )
        proph.fit(train_data)
        y_pred = proph.predict(X_ho)['yhat']

        y_pred = np.clip(y_pred, 0, system_recorded_max)
        darkness_mask = (~((X_ho['proportion_daytime']==0) & (X_ho['global_tilted_irradiance']==0))).astype(int)
        y_pred = y_pred * np.array(darkness_mask)

        error = PostRun.custom_error(ho_data['y'], y_pred)
        errors.append(error)

    return f"{hyp}", errors


changepoint_prior_scale = [0.1]
n_changepoints = [10]
seasonality_prior_scale = [20]
holidays_prior_scale = [2]

for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f"System {system_id}, {reader_type}")

    prerun = PreRun(system_id=system_id, meter_or_inverter=reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
    prerun.fill_missing_hours()
    prerun.add_weather_features_only()
    prerun.good_end_days_naive(7)
    prerun.tts_of_data_using_end_days()
    all_data = prerun.amended_data.copy()
    system_recorded_max = prerun.data['energy'].max()

    all_data.rename(columns={'time': 'ds', 'energy': 'y'}, inplace=True)

    first_day = prerun.good_days.loc[0, 'date']
    remove_until = first_day + timedelta(days=730)
    train_dates = prerun.train_dates.loc[prerun.train_dates['date'] >= remove_until].reset_index(drop=True)
    n = int(len(train_dates) / 200)
    some_train_dates = train_dates[::n]

    splits = []
    for pred_date in some_train_dates['date']:
        train_mask = (all_data['ds'] < pred_date - timedelta(days=1))
        ho_mask = ((all_data['ds'] >= pred_date) & (all_data['ds'] < pred_date + timedelta(days=1)))
        train_data = all_data.loc[train_mask].reset_index(drop=True)
        ho_data = all_data.loc[ho_mask].reset_index(drop=True)
        splits.append((pred_date, train_data, ho_data))

    hyperparams = list(product(changepoint_prior_scale, n_changepoints, seasonality_prior_scale, holidays_prior_scale))

    results = Parallel(n_jobs=4, backend='loky')(
        delayed(_run_one_hyp)(hyp, splits, system_recorded_max)
        for hyp in hyperparams
    )

    results_dict = {key: errors for key, errors in results}
    all_errors_df = pd.DataFrame(results_dict)
    all_errors_df.to_csv(f'prophet_errors_mult/{system_id}_{reader_type}_prophet_errors.csv', index=False)
    print(f"  Saved.")

print("Done!")

System 10, None
  Saved.
System 50, None
  Saved.
System 51, None
  Saved.
Done!


In [5]:
#System 10
print('System 10, None, Prophet')
errors = pd.read_csv('prophet_errors_mult/10_None_prophet_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()
#System 50
print('System 50, None, Prophet')
errors = pd.read_csv('prophet_errors_mult/50_None_prophet_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()
#System 51
print('System 51, None, Prophet')
errors = pd.read_csv('prophet_errors_mult/51_None_prophet_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    

System 10, None, Prophet
recorded system max: 1.185825
  Hyperparameters: (0.1, 10, 20, 2)
     mean: 0.029863827886997738, median: 0.0152002897712042, min: 0.0012364961688, max: 0.1827959221335068, std: 0.034978019319069034

System 50, None, Prophet
recorded system max: 7.072975
  Hyperparameters: (0.1, 10, 20, 2)
     mean: 1.2535584612518667, median: 0.633533997007382, min: 0.0162326569622438, max: 7.4539729713356, std: 1.5672983377670107

System 51, None, Prophet
recorded system max: 7.2368749999999995
  Hyperparameters: (0.1, 10, 20, 2)
     mean: 0.9606655090359852, median: 0.5450524367040546, min: 0.0616024801351069, max: 5.326777360042748, std: 1.0302791982531103
